# Fine-Tune Whisper for Moroccan Darija (TDD-01)

Trains `openai/whisper-small` on **DODa** (`atlasia/DODa-audio-dataset`) for the
Smart Airport Wayfinding Assistant — the project's owned ML contribution.

**Runs on a free GPU**, either way:
- **Browser Colab:** Runtime -> Change runtime type -> **T4 GPU**.
- **VS Code (Google Colab extension):** Select Kernel -> Colab -> Auto Connect (T4).

Just run the cells top to bottom. The heavy lifting is in the repo's `src/`
scripts — this notebook only orchestrates them on the GPU machine.

> The GPU machine is **temporary**: when the session ends it is wiped. Step 7
> pushes the trained model to the Hugging Face Hub so it survives.

## 1. Check the GPU
Confirms a GPU is attached. If this errors or shows no GPU, enable it first (see above).

In [1]:
!nvidia-smi

Sat Jun 20 23:20:49 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   40C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 2. Get the code
Clone the repo and move into the ASR package (all `src.` modules run from here).

In [5]:
# Clone once per session; pull on re-run so you get fixes from main.
import os
from pathlib import Path

if not os.path.isdir('ML-21-Flughafen'):
    !git clone https://github.com/Oussamawork/ML-21-Flughafen.git
else:
    !git -C ML-21-Flughafen pull --ff-only origin main

%cd ML-21-Flughafen/asr_finetuning

# Hotfix until merged: DODa has ~22 rows with null darija_Arab_new transcripts.
# Without filtering, _grouped_split crashes sorting None vs str.
_data = Path('src/data.py')
_src = _data.read_text()
if '_drop_empty_transcripts' not in _src:
    _insert = '''
def _drop_empty_transcripts(dataset, text_column: str, name: str):
    """Remove rows with missing or blank transcripts (DODa has ~22 nulls)."""
    before = len(dataset)

    def keep(text):
        return text is not None and str(text).strip() != ""

    filtered = dataset.filter(keep, input_columns=[text_column])
    dropped = before - len(filtered)
    if dropped:
        print(f"[data] Dropped {dropped} row(s) with empty '{text_column}' in {name}.")
    return filtered


'''
    _src = _src.replace('def _grouped_split', _insert + 'def _grouped_split')
    _src = _src.replace(
        '    raw["train"] = load_dataset(d["name"], split=d["train_split"], **load_kwargs)\n\n    eval_split',
        '    raw["train"] = load_dataset(d["name"], split=d["train_split"], **load_kwargs)\n'
        '    raw["train"] = _drop_empty_transcripts(raw["train"], d["text_column"], d["name"])\n\n'
        '    eval_split',
    )
    _src = _src.replace(
        '            raw["eval"] = load_dataset(d["name"], split=eval_split, **load_kwargs)\n        except',
        '            raw["eval"] = load_dataset(d["name"], split=eval_split, **load_kwargs)\n'
        '            raw["eval"] = _drop_empty_transcripts(\n'
        '                raw["eval"], d["text_column"], d["name"]\n'
        '            )\n        except',
    )
    _data.write_text(_src)
    print('[patch] Applied DODa null-transcript filter to src/data.py')
else:
    print('[patch] src/data.py already includes _drop_empty_transcripts')

!pwd

Cloning into 'ML-21-Flughafen'...
remote: Enumerating objects: 146, done.
remote: Counting objects: 100% (146/146), done.
remote: Compressing objects: 100% (111/111), done.
remote: Total 146 (delta 49), reused 119 (delta 28), pack-reused 0 (from 0)
Receiving objects: 100% (146/146), 90.64 KiB | 15.11 MiB/s, done.
Resolving deltas: 100% (49/49), done.
/content/ML-21-Flughafen/asr_finetuning
/content/ML-21-Flughafen/asr_finetuning


## 3. Install dependencies
Installs transformers/datasets/etc. Colab already ships PyTorch + CUDA.

In [6]:
!pip install -q -r requirements.txt

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 120.5 MB/s eta 0:00:00


## 4. Log in to Hugging Face
DODa is **gated**. First, on huggingface.co, open the
[dataset page](https://huggingface.co/datasets/atlasia/DODa-audio-dataset) and
click **Agree / Access** while logged in. Then create a token at
`huggingface.co/settings/tokens` (role: *read*, or *write* if you will push the
model in step 7) and paste it below.

In [8]:
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
from pathlib import Path

p = Path('src/data.py')
src = p.read_text()
if '_drop_empty_transcripts' not in src:
    insert = '''
def _drop_empty_transcripts(dataset, text_column: str, name: str):
    before = len(dataset)
    def keep(text):
        return text is not None and str(text).strip() != ""
    filtered = dataset.filter(keep, input_columns=[text_column])
    dropped = before - len(filtered)
    if dropped:
        print(f"[data] Dropped {dropped} row(s) with empty '{text_column}' in {name}.")
    return filtered


'''
    src = src.replace('def _grouped_split', insert + 'def _grouped_split')
    src = src.replace(
        '    raw["train"] = load_dataset(d["name"], split=d["train_split"], **load_kwargs)\n\n    eval_split',
        '    raw["train"] = load_dataset(d["name"], split=d["train_split"], **load_kwargs)\n'
        '    raw["train"] = _drop_empty_transcripts(raw["train"], d["text_column"], d["name"])\n\n'
        '    eval_split',
    )
    p.write_text(src)
    print('Patched — re-run training cell.')
else:
    print('Already patched.')

## 5. Smoke test (optional, ~a few min)
A tiny 5-step run to prove the whole pipeline works *before* the long job.
It downloads + caches DODa (reused by the real run) and trains on a handful of
samples. Success = it finishes without error and prints a WER number.

In [9]:
!python -m src.train --config config/doda_darija.yaml \
    --dataset.max_train_samples 64 --dataset.max_eval_samples 16 \
    --training.max_steps 5 --training.eval_steps 5 --training.save_steps 5 \
    --training.warmup_steps 1 --training.logging_steps 1 \
    --training.output_dir ./outputs/smoke

Loading base model: openai/whisper-small (language=arabic, task=transcribe)
preprocessor_config.json: 100% 185k/185k [00:00<00:00, 41.9MB/s]
tokenizer_config.json: 100% 283k/283k [00:00<00:00, 63.3MB/s]
vocab.json: 100% 836k/836k [00:00<00:00, 13.4MB/s]
tokenizer.json: 100% 2.48M/2.48M [00:00<00:00, 22.3MB/s]
merges.txt: 100% 494k/494k [00:00<00:00, 4.69MB/s]
normalizer.json: 100% 52.7k/52.7k [00:00<00:00, 15.7MB/s]
added_tokens.json: 100% 34.6k/34.6k [00:00<00:00, 14.0MB/s]
special_tokens_map.json: 100% 2.19k/2.19k [00:00<00:00, 7.52MB/s]
config.json: 100% 1.97k/1.97k [00:00<00:00, 5.24MB/s]
model.safetensors: 100% 967M/967M [00:07<00:00, 135MB/s] 
Loading weights: 100% 479/479 [00:00<00:00, 5109.63it/s]
generation_config.json: 100% 3.87k/3.87k [00:00<00:00, 13.4MB/s]
Loading and preprocessing dataset...
README.md: 100% 5.36k/5.36k [00:00<00:00, 1.66MB/s]
data/train-00000-of-00005.parquet: 100% 333M/333M [00:11<00:00, 28.7MB/s]
data/train-00001-of-00005.parquet: 100% 279M/279M [00:08<

## 6a. Baseline: evaluate the *un-tuned* model
Measure `whisper-small` BEFORE fine-tuning on the DODa eval split. Note this
WER/CER — the improvement over it is your headline result.

In [ ]:
!python -m src.evaluate_model --config config/doda_darija.yaml \
    --model.name openai/whisper-small

## 6b. Fine-tune (the long step, ~1-3 h on a T4)
Trains on DODa with the settings in `config/doda_darija.yaml`
(`max_steps=3000`, `batch=16`, fp16, best-checkpoint-by-WER). Eval WER prints
every 300 steps and should trend **down**. Leave the tab open.

**If you hit a CUDA out-of-memory error**, lower the batch size and compensate
with accumulation (uncomment the second line):

In [ ]:
!python -m src.train --config config/doda_darija.yaml

# OOM-safe variant (same effective batch size, less memory):
# !python -m src.train --config config/doda_darija.yaml \
#     --training.per_device_train_batch_size 8 --training.gradient_accumulation_steps 2

## 6c. Evaluate the fine-tuned model
Same eval split as the baseline -> compare the two WER/CER numbers. This delta
is the graded ML contribution. `evaluate_model` also prints a few REF/PRED
examples for the report.

In [ ]:
!python -m src.evaluate_model --config config/doda_darija.yaml \
    --model.name ./outputs/whisper-small-doda-darija

## 7. Save the model to the Hugging Face Hub
The trained model lives only on this temporary machine. Push it to your account
so it is kept and can be loaded by the backend later. Needs a **write** token
(step 4). Change the repo id to your username.

In [ ]:
from transformers import WhisperForConditionalGeneration, WhisperProcessor

CKPT = './outputs/whisper-small-doda-darija'
REPO = 'Oussamawork/whisper-small-darija'  # <-- change to your HF username/repo

WhisperForConditionalGeneration.from_pretrained(CKPT).push_to_hub(REPO)
WhisperProcessor.from_pretrained(CKPT).push_to_hub(REPO)
print('Pushed to https://huggingface.co/' + REPO)

## 8. Quick listen test (optional)
Transcribe one DODa clip with the fine-tuned model and compare to the reference
transcript — a sanity check that it actually learned Darija.

In [ ]:
from datasets import load_dataset, Audio
from src.transcribe import WhisperTranscriber

sample = load_dataset('atlasia/DODa-audio-dataset', split='train[:1]')\
    .cast_column('audio', Audio(sampling_rate=16000))[0]

tr = WhisperTranscriber('./outputs/whisper-small-doda-darija')
pred = tr.transcribe(sample['audio']['array'], sampling_rate=16000)
print('REF :', sample['darija_Arab_new'])
print('PRED:', pred)